# Finish the Quoridor curriculum
Resume the existing small, residual128 and residual256 seed-1 runs to **200 total iterations**, preserving their training settings and threefold cutoff. No new repetition experiment or GPU sweep runs here.

Upload this notebook to Colab, select a GPU runtime, and run the cells in order. Upload the refreshed `colab-source.zip` from this workspace when prompted. Your original `latest.pt` checkpoints must still be in Drive; exported bots alone cannot resume training.

With the existing 25-iteration curriculum interval:

| Iterations | Unlocked race stage | Walls per player |
| --- | --- | --- |
| 1–25 | race-0 | 0 |
| 26–50 | race-1 | 2 |
| 51–75 | race-2 | 5 |
| 76–200 | race-3 | 10 |

Earlier tasks remain mixed in and standard full-wall starts remain present. Reaching stage 3 means completing the schedule, not proven mastery; inspect full-wall evaluation wins. All three runs continue sequentially, and rerunning safely resumes unfinished work.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io, os, subprocess, sys
uploaded = files.upload()
source = uploaded.get('colab-source.zip')
if source is None:
    raise ValueError('Upload colab-source.zip generated by training/prepare_colab.py')
root = Path('/content/Quoridoraner')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    for entry in archive.infolist():
        target = (root / entry.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(root)
os.chdir(root)
# Keep Colab's CUDA-enabled PyTorch; do not install the Mac requirements pin.
import torch, numpy
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('PyTorch', torch.__version__, 'CUDA', torch.version.cuda)
print(torch.cuda.get_device_properties(0))
subprocess.run([sys.executable, 'training/native/build_direct.py'], check=True)


## Mount the Drive containing the original runs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/quoridor-experiments')
RUN_ROOT.mkdir(parents=True, exist_ok=True)


## Check the selected checkpoints before starting
Change the list only if you want to include other existing runs. This does not create fresh runs.


In [ ]:
import json
run_names = ['small-bf16-seed1', 'residual128-bf16-seed1', 'residual256-bf16-seed1']
total_iterations = 200
for name in run_names:
    directory = RUN_ROOT / name
    if not (directory / 'latest.pt').is_file():
        raise FileNotFoundError(f'Missing resumable checkpoint: {directory / "latest.pt"}')
    cfg = json.loads((directory / 'config.json').read_text())
    if cfg.get('curriculum') != 'races' or cfg.get('curriculum_interval') != 25:
        raise ValueError(f'{name}: expected the original scheduled 25-iteration curriculum')
    if cfg.get('training_repetition_limit', 3) != 3:
        raise ValueError(f'{name}: expected original threefold training cutoff')
    if cfg.get('precision') == 'bf16' and not torch.cuda.is_bf16_supported():
        raise RuntimeError('These saved runs require a BF16-capable CUDA GPU')
    if cfg.get('iterations', 0) > total_iterations:
        raise ValueError(f'{name}: saved iteration budget exceeds the selected total')
    print(name, 'architecture:', cfg['architecture'], 'seed:', cfg['seed'],
          'games:', cfg['games'], 'updates:', cfg['updates'])


## Continue training
Progress is displayed at most every two seconds. Full console logs are written locally and copied to Drive when each process exits. The trainer saves checkpoints directly to Drive. Interrupting this cell requests a graceful save; wait for the process to exit before rerunning. Runtime deletion can only recover the last completed checkpoint.

Resuming restores optimizer, replay, RNG, active games, curriculum and precision. The command extends only the total iteration budget and uses CUDA. It preserves the saved checkpoint cadence and training settings.


In [ ]:
import signal, time, shutil
for name in run_names:
    directory = RUN_ROOT / name
    command = [sys.executable, '-u', 'training/train.py', '--resume', str(directory),
               '--iterations', str(total_iterations), '--device', 'cuda', '--plain']
    print('START', name, flush=True)
    local_log = Path('/content') / f'{name}-continuation.log'
    last_display = 0
    try:
        with local_log.open('a') as log, subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    log.write(line)
                    now = time.monotonic()
                    if now - last_display >= 2:
                        print(line, end='', flush=True)
                        last_display = now
                if process.wait():
                    raise RuntimeError(f'{name} failed; inspect {local_log}')
            except KeyboardInterrupt:
                if process.poll() is None:
                    process.send_signal(signal.SIGINT)
                print('Waiting for checkpoint save...', flush=True)
                for line in process.stdout:
                    log.write(line)
                process.wait()
                raise
    finally:
        if local_log.exists():
            shutil.copy2(local_log, directory / 'continuation.log')
    print('FINISHED', name, flush=True)


## Review full-wall evaluations
Goal wins matter more than training loss or cutoff-derived half-points. Compare final and earlier monitoring evaluations within each arm.


In [ ]:
import pandas as pd
rows = []
for name in run_names:
    path = RUN_ROOT / name / 'summary.json'
    if path.exists():
        for row in json.loads(path.read_text()):
            if row['phase'] == 'eval' and row.get('suite') in ('monitor', 'heldout'):
                rows.append({'run': name, **row})
display(pd.DataFrame(rows)[['run', 'mode', 'iteration', 'suite', 'games',
                            'wins', 'losses', 'cutoffs', 'score']])


## Download results for local head-to-head comparisons
Run after training completes. Includes both bot exports and JSON reports; resumable checkpoints remain on Drive.


In [ ]:
archive = Path('/content/quoridor-curriculum-results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in run_names:
        directory = RUN_ROOT / name
        for path in [directory / 'selfplay.pt', directory / 'negamax.pt', *directory.glob('*.json')]:
            if path.is_file():
                z.write(path, f'{name}/{path.name}')
files.download(str(archive))
